In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


In [2]:
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

In [3]:
data = pd.read_csv("cleaned_heart.csv")

print("Shape:", data.shape)
data.head()

Shape: (920, 20)


,id,age,trestbps,chol,fbs,thalch,exang,oldpeak,num,sex_Male,dataset_Hungary,dataset_Switzerland,dataset_VA Long Beach,cp_atypical angina,cp_non-anginal,cp_typical angina,restecg_normal,restecg_st-t abnormality,slope_flat,slope_upsloping
0,1,63,145.0,233.0,True,150.0,False,2.3,0,True,False,False,False,False,False,True,False,False,False,False
1,2,67,160.0,286.0,False,108.0,True,1.5,2,True,False,False,False,False,False,False,False,False,True,False
2,3,67,120.0,229.0,False,129.0,True,2.6,1,True,False,False,False,False,False,False,False,False,True,False
3,4,37,130.0,250.0,False,187.0,False,3.5,0,True,False,False,False,False,True,False,True,False,False,False
4,5,41,130.0,204.0,False,172.0,False,1.4,0,False,False,False,False,True,False,False,False,False,False,True


In [4]:
data['risk'] = data['num'].apply(lambda x: 1 if x > 0 else 0)

data['risk'].value_counts()

risk
1    509
0    411
Name: count, dtype: int64

In [5]:
drop_cols = ['num', 'risk']

if 'id' in data.columns:
    drop_cols.append('id')

X = data.drop(columns=drop_cols)
y = data['risk']

print("Features:", X.columns.tolist())

Features: ['age', 'trestbps', 'chol', 'fbs', 'thalch', 'exang', 'oldpeak', 'sex_Male', 'dataset_Hungary', 'dataset_Switzerland', 'dataset_VA Long Beach', 'cp_atypical angina', 'cp_non-anginal', 'cp_typical angina', 'restecg_normal', 'restecg_st-t abnormality', 'slope_flat', 'slope_upsloping']


In [6]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler 🔥
joblib.dump(scaler, "scaler.pkl")

print("✅ Scaler saved")

✅ Scaler saved


In [7]:
def generate_sequences(data, labels, timesteps=15):
    X_seq = []
    y_seq = []

    for i in range(len(data)):
        row = data.iloc[i].values
        sequence = []

        for t in range(timesteps):
            step = row.copy()

            # progression
            step = step * (1 + t * 0.03)

            # noise
            step += np.random.normal(0, 0.01, size=row.shape)

            sequence.append(step)

        X_seq.append(sequence)
        y_seq.append(labels.iloc[i])

    return np.array(X_seq), np.array(y_seq)

X_lstm, y_lstm = generate_sequences(X_scaled, y)

print("Shape:", X_lstm.shape)

AttributeError: 'numpy.ndarray' object has no attribute 'iloc'